# Lab 07 Challenge: worked solution

For the instructor. Each task uses only what Lab 07 showed. The first cells repeat the lab's setup, the heading split,
the index and `search`, so this notebook runs on its own. Our run's values are in the notes (`EMBED_MODEL` from
`models.env`, `text-embedding-3-small`, 13 Oct 2026).

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "JUDGE_MODEL", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
JUDGE_MODEL       gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


The pieces from the lab: the 20 heading chunks (Part B), an index built from them and `search` (Part C).

In [3]:
import re
import numpy as np
import faiss
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

HANDBOOK = ROOT / "data" / "handbook" / "support_handbook.md"
emb = OpenAIEmbeddings(model=settings["EMBED_MODEL"])
by_heading = MarkdownHeaderTextSplitter(headers_to_split_on=[("##", "policy")], strip_headers=False)
sections = [doc for doc in by_heading.split_text(HANDBOOK.read_text(encoding="utf-8")) if "policy" in doc.metadata]
chunks = [{"text": doc.page_content, "source": doc.page_content.split("Source: ")[1].split()[0]} for doc in sections]

def build_index(chunks):
    vectors = np.array(emb.embed_documents([c["text"] for c in chunks]), dtype="float32")
    faiss.normalize_L2(vectors)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index

index = build_index(chunks)

def search(query, k=3, index=index, chunks=chunks):
    vector = np.array([emb.embed_query(query)], dtype="float32")
    faiss.normalize_L2(vector)
    scores, ids = index.search(vector, k)
    return [(float(s), chunks[i]) for s, i in zip(scores[0], ids[0])]

print(index.ntotal, "chunks")

20 chunks


## Task 1: Reword question 4

Three rewordings that keep the rep's meaning: the parcel is on its way, and the customer wants it delivered somewhere
else. The cell searches all 20 chunks to print both files' scores.

In [4]:
REWORDINGS = [
    "The parcel has shipped. Can we redirect it to her holiday address?",          # the original
    "Can we redirect a parcel that is already in transit?",
    "Her order is on its way. Can we change the delivery address now?",
    "It has already shipped. Can the courier take it to a different address?",
]
for q in REWORDINGS:
    hits = search(q, k=20)
    by_file = {c["source"]: s for s, c in hits}
    print(f"first: {hits[0][1]['source']:26} | international {by_file['policy_international.md']:.4f}"
          f"  shipping {by_file['policy_shipping.md']:.4f} | {q}")

first: policy_shipping.md         | international 0.3680  shipping 0.3875 | The parcel has shipped. Can we redirect it to her holiday address?


first: policy_cancellations.md    | international 0.3212  shipping 0.3008 | Can we redirect a parcel that is already in transit?


first: policy_shipping.md         | international 0.3289  shipping 0.3720 | Her order is on its way. Can we change the delivery address now?
first: policy_shipping.md         | international 0.3351  shipping 0.3853 | It has already shipped. Can the courier take it to a different address?


None of the three rewordings puts `policy_international.md` first. *Redirect a parcel that is already in transit*,
close to the policy's own words (*We do not redirect a parcel in transit*), lifts the international policy above
shipping (0.3212 against 0.3008), but the cancellations policy comes first. *Change the delivery address* and *take it
to a different address* keep shipping first (0.3720 and 0.3853, against 0.3289 and 0.3351). The words that moved the
ranking, *redirect* and *in transit*, are the policy's own. An earlier version of this solution asked *Can a parcel be rerouted to
another country after it ships?*, and that put the international policy first (0.4604 against 0.4213, 12 Oct 2026),
but it is a different question: the rep never said the address is abroad.

A copilot cannot rely on this. Reps write the way customers write, and the words that move the ranking are the
policy's own, which a rep would use only after reading the policy. The fix belongs in the search (keyword or hybrid
search, smaller chunks, a reranker) or in what the copilot does with three chunks: Lab 08's copilot reads all three,
not only the first.

## Task 2 (optional): Split by heading, then by size

Cut each section from the heading split into pieces of at most 200 characters with no overlap, and copy the section's
source file onto every piece.

In [5]:
by_size = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)
small_chunks = []
for section, chunk in zip(sections, chunks):
    for piece in by_size.split_text(section.page_content):
        small_chunks.append({"text": piece, "source": chunk["source"]})

RULE = "Duplicate charges: if two invoices exist for the same album on the same day, refund the later invoice in full."
print(len(small_chunks), "chunks; every chunk has a source:", all(c["source"] for c in small_chunks))
holding = [c for c in small_chunks if RULE in c["text"]]
print("chunks holding the whole rule:", len(holding), "| source:", [c["source"] for c in holding])

50 chunks; every chunk has a source: True
chunks holding the whole rule: 1 | source: ['policy_refunds.md']


The rule's piece no longer contains the `Source:` line, but the source travels with it as data, which is what a
citation needs. Search the bought-twice question against the small chunks.

In [6]:
small_index = build_index(small_chunks)
question = "She bought the same album twice by mistake. Can she get her money back?"
for score, chunk in search(question, index=small_index, chunks=small_chunks):
    print(f"{score:.4f}  {chunk['source']:26} holds the rule: {RULE in chunk['text']}")

0.5060  policy_refunds.md          holds the rule: True
0.4722  policy_damaged_media.md    holds the rule: False
0.4567  policy_refunds.md          holds the rule: False


50 chunks, every one with a source, the rule whole in one chunk from `policy_refunds.md`, and the bought-twice question finds it first with 0.5060, higher than the whole refunds chunk scored in the lab (0.4423), because the small chunk is about one thing. The price: the third hit is another piece of the same policy, so three results cover fewer policies, and a rule whose sentences were cut apart loses its context.

## Task 3 (optional): Test a threshold on new questions

Lab 07 Part F's best-looking threshold was 0.35: one covered question refused (Bitcoin) and one uncovered question answered (instalments), on the 18 questions it was chosen on. Four new questions, written before running anything: two that a policy answers and two that no policy
answers.

In [7]:
THRESHOLD = 0.35
NEW = [
    ("Can a customer get a copy of the data we store about her?", "policy_privacy.md"),
    ("A customer has three open tickets. Who should take over?", "policy_escalations.md"),
    ("Do we offer a free trial of a streaming subscription?", None),
    ("Can the customer collect her order from the warehouse in person?", None),
]
mistakes = 0
for q, answer in NEW:
    score, chunk = search(q, k=1)[0]
    decision = "answer" if score >= THRESHOLD else "refuse"
    right = (decision == "answer") == (answer is not None)
    mistakes += not right
    print(f"{score:.4f}  {decision:6}  {'covered' if answer else 'NOT covered':11}  top {chunk['source']:26} "
          f"{'right' if right else 'MISTAKE'}  | {q}")
print("mistakes on the new questions:", mistakes, "of", len(NEW))

0.5186  answer  covered      top policy_privacy.md          right  | Can a customer get a copy of the data we store about her?


0.4919  answer  covered      top policy_escalations.md      right  | A customer has three open tickets. Who should take over?


0.2996  refuse  NOT covered  top policy_library.md          right  | Do we offer a free trial of a streaming subscription?
0.3479  refuse  NOT covered  top policy_shipping.md         right  | Can the customer collect her order from the warehouse in person?
mistakes on the new questions: 0 of 4


0.35 makes no mistake on the four new questions, but *collect her order from the warehouse* (not covered) scores 0.3479, 0.002 under the threshold: a slightly different wording could let it through. Four questions cannot confirm a threshold. The decision to say "not covered" stays with the model reading the passages, checked in code (Lab 08).